<a href="https://colab.research.google.com/github/SahilBhragudev/Machine-Learning-Projects-Models/blob/machineLearning/Fake_News_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# Import the Dependencies
import pandas as pd
import numpy as np
import re #regular expression used for searching words
from nltk.corpus import stopwords # natural language tool kit used to search stop words
from nltk.stem.porter import PorterStemmer # Gives the root word for a particular word
from sklearn.feature_extraction.text import TfidfVectorizer # Used to convert the text into feature
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


In [3]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [9]:
# Printing the stopwords in English
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

Data Pre processing

In [11]:
# Loading the dataset to pandas dataframe
news_dataset = pd.read_csv('/content/FA-KES-Dataset.csv',encoding='latin1')

In [13]:
news_dataset.shape

(804, 7)

In [14]:
news_dataset.head()

,unit_id,article_title,article_content,source,date,location,labels
0,1914947530,Syria attack symptoms consistent with nerve ag...,Wed 05 Apr 2017 Syria attack symptoms consiste...,nna,4/5/2017,idlib,0
1,1914947532,Homs governor says U.S. attack caused deaths b...,Fri 07 Apr 2017 at 0914 Homs governor says U.S...,nna,4/7/2017,homs,0
2,1914947533,Death toll from Aleppo bomb attack at least 112,Sun 16 Apr 2017 Death toll from Aleppo bomb at...,nna,4/16/2017,aleppo,0
3,1914947534,Aleppo bomb blast kills six Syrian state TV,Wed 19 Apr 2017 Aleppo bomb blast kills six Sy...,nna,4/19/2017,aleppo,0
4,1914947535,29 Syria Rebels Dead in Fighting for Key Alepp...,Sun 10 Jul 2016 29 Syria Rebels Dead in Fighti...,nna,7/10/2016,aleppo,0


In [15]:
# check if some values are missing in the datset
news_dataset.isnull().sum()

,0
unit_id,0
article_title,0
article_content,0
source,0
date,0
location,0
labels,0


In [17]:
# Separting data and label
X = news_dataset.drop(columns='labels',axis = 1)
Y = news_dataset['labels']

In [18]:
print(X)
print(Y)

        unit_id  ...    location
0    1914947530  ...       idlib
1    1914947532  ...        homs
2    1914947533  ...      aleppo
3    1914947534  ...      aleppo
4    1914947535  ...      aleppo
..          ...  ...         ...
799  1965511221  ...      aleppo
800  1965511222  ...      aleppo
801  1965511224  ...      aleppo
802  1965511226  ...       idlib
803  1965511231  ...  deir ezzor

[804 rows x 6 columns]
0      0
1      0
2      0
3      0
4      0
      ..
799    1
800    1
801    0
802    1
803    1
Name: labels, Length: 804, dtype: int64


In [26]:
news_dataset['content'] = news_dataset['article_content'] + ' ' + news_dataset['location']

In [27]:
print(news_dataset['content'])

0      Wed 05 Apr 2017 Syria attack symptoms consiste...
1      Fri 07 Apr 2017 at 0914 Homs governor says U.S...
2      Sun 16 Apr 2017 Death toll from Aleppo bomb at...
3      Wed 19 Apr 2017 Aleppo bomb blast kills six Sy...
4      Sun 10 Jul 2016 29 Syria Rebels Dead in Fighti...
                             ...                        
799    28-08-2016 Turkish Bombardment Kills 20 Civili...
800    17-08-2016 Martyrs as Terrorists Shell Aleppos...
801    03-08-2016 Chemical Attack Kills Five Syrians ...
802    01-08-2016 5 Killed as Russian Military Choppe...
803    April 6 2017 Syrian Army Kills 48 ISIL Terrori...
Name: content, Length: 804, dtype: object


Stemming

In [19]:
# Stemming is the process of reducing a word to it's root word
# Eg : actor , acting , actress -----> root word is 'act'

port_stem = PorterStemmer()

In [30]:
def stemming(content):
  stemmed_content = re.sub('[^a-zA-Z],','',content)
  stemmed_content = stemmed_content.lower()
  stemmed_content = stemmed_content.split()
  stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
  stemmed_content = ' '.join(stemmed_content)
  return stemmed_content

In [31]:
news_dataset['content'] = news_dataset['content'].apply(stemming)

In [32]:
print(news_dataset['content'])

0      wed 05 apr 2017 syria attack symptom consist n...
1      fri 07 apr 2017 0914 hom governor say u.s. att...
2      sun 16 apr 2017 death toll aleppo bomb attack ...
3      wed 19 apr 2017 aleppo bomb blast kill six syr...
4      sun 10 jul 2016 29 syria rebel dead fight key ...
                             ...                        
799    28-08-2016 turkish bombard kill 20 civilian sy...
800    17-08-2016 martyr terrorist shell aleppo salah...
801    03-08-2016 chemic attack kill five syrian alep...
802    01-08-2016 5 kill russian militari chopper sho...
803    april 6 2017 syrian armi kill 48 isil terroris...
Name: content, Length: 804, dtype: object


In [33]:
# Separting data and label
X = news_dataset['content'].values
Y = news_dataset['labels'].values

In [34]:
print(X)
print(Y)

['wed 05 apr 2017 syria attack symptom consist nerv agent use who. victim suspect chemic attack syria appear show symptom consist reaction nerv agent world health organ said wednesday. "some case appear show addit sign consist exposur organophosphoru chemic categori chemic includ nerv agents" said statement put death toll least 70. unit state said death caus sarin nerv ga drop syrian aircraft. russia said believ poison ga leak rebel chemic weapon depot struck syrian bombs. sarin organophosporu compound nerv agent. chlorin mustard ga also believ use past syria not. russian defenc ministri spokesman say agent use attack said rebel use chemic weapon aleppo last year. said like kind chemic use attack suffer appar extern injuri die rapid onset similar symptom includ acut respiratori distress. said expert turkey give guidanc overwhelm health worker idlib diagnosi treatment patient medicin atropin antidot type chemic exposur steroid symptomat treatment sent. u.n. commiss inquiri human right s

In [35]:
# convert the text data into meaningful numbers so computer can underdstand
# use the vectorization function

vectorizer = TfidfVectorizer()
vectorizer.fit(X)

X = vectorizer.transform(X)

In [42]:
print(Y)

[0 0 0 0 0 0 0 0 1 0 1 1 1 1 0 0 1 1 0 0 0 0 1 1 1 1 1 0 0 1 0 0 1 0 0 1 1
 1 1 1 1 1 1 0 1 1 1 1 1 1 0 0 1 1 0 0 1 0 1 1 1 1 0 1 1 1 0 0 0 1 0 1 0 0
 1 0 1 1 0 0 0 0 0 0 1 1 1 1 0 0 0 0 1 1 0 1 1 1 1 1 0 1 0 0 1 1 1 0 0 1 1
 1 1 1 1 1 0 1 1 1 1 1 1 1 1 0 1 0 0 1 0 1 0 1 0 1 1 1 1 1 0 1 1 1 0 1 0 0
 1 1 0 0 0 1 1 1 1 0 1 1 1 1 1 0 0 1 1 0 1 0 1 0 0 0 0 1 1 0 0 1 0 1 1 0 0
 1 0 1 0 0 0 0 1 0 0 1 1 0 0 0 0 0 1 0 0 0 1 0 1 1 1 1 1 0 0 0 1 1 0 1 0 1
 0 0 0 0 0 1 0 0 0 0 1 1 0 1 1 1 0 1 1 0 0 0 1 1 0 1 1 0 0 1 1 0 1 1 0 1 1
 0 1 1 0 0 1 1 1 1 0 0 0 0 1 1 0 1 0 0 0 1 0 0 1 0 0 1 1 1 1 1 1 1 1 1 1 0
 1 0 0 0 1 1 1 0 1 1 1 1 1 0 0 1 0 1 1 0 1 0 0 0 1 0 1 1 1 1 1 0 0 0 0 1 1
 0 1 1 1 0 0 1 0 0 1 1 1 1 0 1 1 0 0 0 1 0 0 0 0 0 1 0 1 0 1 0 0 1 0 0 0 1
 1 0 1 0 1 1 0 0 0 0 1 0 1 0 1 1 1 1 1 0 1 1 1 1 0 1 0 0 1 0 0 0 0 1 1 0 1
 1 1 0 1 1 0 0 1 1 0 0 0 1 1 1 0 0 0 1 1 1 0 0 0 0 1 1 1 1 1 0 0 1 1 0 1 0
 0 0 0 1 1 0 0 0 0 1 0 1 0 0 0 0 0 1 0 0 0 0 1 1 1 1 0 0 1 0 0 1 1 1 1 1 0
 0 1 0 0 1 1 0 0 0 1 0 1 

In [43]:
# split data into training and test data
X_train, X_test ,Y_train,Y_test = train_test_split(X,Y,test_size = 0.2 , stratify = Y,random_state=2)

In [39]:
# Training the model : logistic Regression Model
model = LogisticRegression()

In [44]:
model.fit(X_train, Y_train)

LogisticRegression()

In [45]:
# Evaluation of model
X_train_prediction = model.predict(X_train)
training_data_accuracy = accuracy_score(X_train_prediction,Y_train)

In [48]:
print('Accuracy score of training data is : ',training_data_accuracy)

Accuracy score of training data is :  0.8880248833592534


In [49]:
X_test_prediction = model.predict(X_test)
test_data_accuracy = accuracy_score(X_test_prediction , Y_test)

In [50]:
print('Accuracy score of test data is : ',test_data_accuracy)

Accuracy score of test data is :  0.5341614906832298


In [53]:
# Making a predictive system
X_new = X_test[0]
prediction = model.predict(X_new)
if prediction == 0:
  print('Real news')
else:
  print('Fake news')


Real news


In [54]:
print(Y_test[0])

0
